In [2]:
import numpy as np
import pandas as pd
import sys
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project")

from functions_v2 import *

In [ ]:
edges, n_vertices, edge_weights = load_graph("../../data/raw/aves-weaver-social-02.edges")

In [ ]:
edges

In [ ]:
all_nodes = sorted(set(v for e in edges for v in e))
node_map = {old: new for new, old in enumerate(all_nodes)}
edges = [(node_map[i], node_map[j]) for (i, j) in edges]
n_vertices = len(all_nodes)

print("Remapped edges:", edges)
print("n_vertices:", n_vertices)

- graph had node labels from 25-29, not zero-index
- remapped them so they become zero indexed - 0 to 4

In [ ]:
# visualise the graph
visualize_graph_2d(edges, n_vertices, edge_weights=None,title="Aves", save_as=None)

In [ ]:
A,D,L = build_graph_matrices(edges,n_vertices)

In [ ]:
print(A.toarray())

In [ ]:
print(D.toarray())

In [ ]:
print(L.toarray())

# Phase 2

In [ ]:
lambda_min = compute_lambda_min(L, D)

In [ ]:
L_sigma = build_shifted_laplacian(L, D, lambda_min, sigma_squared=0.25)
L_sigma

In [ ]:
print(L_sigma.toarray())

# Phase 3

In [ ]:
# build incidence matrix
B = build_incidence_matrix(edges, n_vertices)

In [ ]:
print(B.toarray())

In [ ]:
# 3a - draw edge noise 
w = draw_edge_noise(edges, seed=42)
w

In [ ]:
f = B @ w

In [ ]:
n_edges = 10

In [ ]:
print(f"{'Vertex':<8}{'Incident edges':<40}{'w values used':<35}{'f_i':<10}")
print("-" * 90)
for v in range(n_vertices):
    incident = [(idx, edges[idx]) for idx in range(n_edges) if v in edges[idx]]
    incident_str = ", ".join([f"e{idx}{pair}" for idx, pair in incident])
    w_vals = [round(w[idx], 4) for idx, _ in incident]
    w_str = " + ".join([str(val) for val in w_vals])
    print(f"{v:<8}{incident_str:<40}{w_str:<35}{f[v]:<10.4f}")

print("\nFull w array:", np.round(w, 4))
print("Full f array:", np.round(f, 4))

In [ ]:
# solve for u
from sksparse.cholmod import cholesky as sparse_cholesky

L_sigma_sparse = L_sigma.tocsc() if hasattr(L_sigma, 'tocsc') else L_sigma
factor = sparse_cholesky(L_sigma_sparse)
u = factor.solve_A(np.sqrt(lambda_min) * f)
print("u =", np.round(u, 4))

# Phase 4

In [ ]:
i_arr = np.array([e[0] for e in edges])
j_arr = np.array([e[1] for e in edges])
k_vals = np.exp((u[i_arr] + u[j_arr]) / 2)

print(f"{'Edge':<10}{'(i,j)':<10}{'u_i':<12}{'u_j':<12}{'k_e':<10}")
for idx, (i, j) in enumerate(edges):
    print(f"e{idx:<9}{str((i,j)):<10}{u[i]:<12.6f}{u[j]:<12.6f}{k_vals[idx]:<10.4f}")

print("\nk_vals =", np.round(k_vals, 4))

In [ ]:
k0 = np.exp((1.8667+1.56)/2)
k5 = np.exp((1.4143+1.56)/2)

In [ ]:
print(k0,k5)

# phase 5

In [ ]:
# Weighted adjacency A_k
A_k = np.zeros((5, 5))
for idx, (i, j) in enumerate(edges):
    k_e = k_vals[idx]
    A_k[i, j] = k_e
    A_k[j, i] = k_e

print("A_k =")
print(np.round(A_k, 4))

In [ ]:
#Weighted degree D_k — row sums of A_k
weighted_degrees = A_k.sum(axis=1)
D_k = np.diag(weighted_degrees)

print("\nD_k =")
print(np.round(D_k, 4))

In [ ]:
# Weighted Laplacian
L_k = D_k - A_k
print("\nL_k =")
print(np.round(L_k, 4))

In [ ]:
# sanity checks for L_k
print("\nRow sums (should be ~0):", np.round(L_k.sum(axis=1), 8))
print("Symmetric?", np.allclose(L_k, L_k.T))

In [ ]:
from scipy.sparse import coo_matrix, diags

def build_weighted_adjacency(edges, n_vertices, k):
    """
    Build the weighted adjacency matrix A_k from edge permeabilities.
    A_k[i,j] = A_k[j,i] = k_e for each edge e=(i,j).
    """
    edges_arr = np.array(edges)
    i_arr = edges_arr[:, 0]
    j_arr = edges_arr[:, 1]
    k_vals = np.asarray(k)

    # both symmetric positions, built in one vectorized pass
    row_idx = np.concatenate([i_arr, j_arr])
    col_idx = np.concatenate([j_arr, i_arr])
    data = np.concatenate([k_vals, k_vals])

    A_k = coo_matrix((data, (row_idx, col_idx)),
                      shape=(n_vertices, n_vertices)).tocsr()
    return A_k

In [ ]:
A_k2 = build_weighted_adjacency(edges, n_vertices, k_vals)

In [ ]:
print(A_k2.toarray())

In [ ]:
import numpy as np
np.array_equal(A_k, A_k2.toarray())

In [ ]:
def build_weighted_degree(A_k):
    """
    Build the weighted degree matrix D_k from a weighted adjacency A_k.
    D_k[i,i] = sum of k_e over all edges incident to vertex i.
    """
    weighted_degrees = np.array(A_k.sum(axis=1)).flatten()
    D_k = diags(weighted_degrees, format='csr')
    return D_k

In [ ]:
D_k2 = build_weighted_degree(A_k2)

In [ ]:
gamma_in = [0]
gamma_out = [2]
p_in, p_out = 1.0, 0.0

boundary = set(gamma_in) | set(gamma_out)
interior = np.array([v for v in range(5) if v not in boundary])

p_boundary = np.zeros(5)
p_boundary[gamma_in[0]] = p_in
p_boundary[gamma_out[0]] = p_out

print("boundary vertices:", sorted(boundary))
print("interior vertices:", interior)

In [ ]:
interior

In [ ]:
L_k

In [ ]:
L_interior = L_k[interior, :][:, interior]
L_interior